## Ingest Sydney Trains Trip Updates (GTFS Realtime v2)

This notebook continuously polls the Trip Update feed from Transport NSW
and sends events to the **Stream_Train_Updates** Eventstream.

Each event contains predicted arrival/departure times and delays for
upcoming stops on active trips.

**Run cadence:** Continuously (same as vehicle position notebook).

In [ ]:
pip install gtfs-realtime-bindings requests azure-servicebus

In [ ]:
import requests
import json
import time
from datetime import datetime
import pytz
from google.transit import gtfs_realtime_pb2
from azure.servicebus import ServiceBusClient, ServiceBusMessage

In [ ]:
# Transport NSW API key
myapikey = ""

# Eventstream custom endpoint connection string
# Find in Fabric portal: open Stream_Train_Updates -> Custom endpoint -> Keys
connection_string = ""

# Polling interval in seconds (feed updates ~every 10-15s)
poll_interval = 15

# Sydney timezone for timestamp conversion
sydney_tz = pytz.timezone("Australia/Sydney")

In [ ]:
def fetch_trip_updates(api_key):
    """Fetch trip updates from the GTFS Realtime v2 feed."""
    url = "https://api.transport.nsw.gov.au/v2/gtfs/realtime/sydneytrains"
    headers = {
        "Authorization": f"apikey {api_key}",
        "Accept": "application/x-google-protobuf"
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)
    return feed

# Test fetch
test_feed = fetch_trip_updates(myapikey)
print(f"Feed timestamp: {datetime.fromtimestamp(test_feed.header.timestamp, tz=sydney_tz)}")
trip_count = sum(1 for e in test_feed.entity if e.HasField('trip_update'))
print(f"Trip updates in feed: {trip_count}")

In [ ]:
def parse_trip_updates(feed):
    """Parse GTFS TripUpdate entities into flat event records.
    
    Each stop_time_update in a TripUpdate becomes a separate event row.
    """
    events = []
    feed_timestamp = feed.header.timestamp

    for entity in feed.entity:
        if not entity.HasField('trip_update'):
            continue

        tu = entity.trip_update
        trip = tu.trip

        for stu in tu.stop_time_update:
            arrival_time = ""
            arrival_delay = 0
            departure_time = ""
            departure_delay = 0

            if stu.HasField('arrival'):
                arrival_delay = stu.arrival.delay
                if stu.arrival.time > 0:
                    arrival_time = datetime.fromtimestamp(
                        stu.arrival.time, tz=sydney_tz
                    ).strftime("%Y-%m-%d %H:%M:%S")

            if stu.HasField('departure'):
                departure_delay = stu.departure.delay
                if stu.departure.time > 0:
                    departure_time = datetime.fromtimestamp(
                        stu.departure.time, tz=sydney_tz
                    ).strftime("%Y-%m-%d %H:%M:%S")

            event = {
                "trip_id": trip.trip_id,
                "route_id": trip.route_id,
                "direction_id": trip.direction_id,
                "schedule_relationship": str(stu.schedule_relationship),
                "stop_id": stu.stop_id,
                "stop_sequence": stu.stop_sequence,
                "arrival_delay": arrival_delay,
                "arrival_time": arrival_time,
                "departure_delay": departure_delay,
                "departure_time": departure_time,
                "timestamp": datetime.fromtimestamp(
                    feed_timestamp, tz=sydney_tz
                ).strftime("%Y-%m-%d %H:%M:%S")
            }
            events.append(event)

    return events

# Test parse
test_events = parse_trip_updates(test_feed)
print(f"Parsed {len(test_events)} stop-time-update events")
if test_events:
    print("Sample event:")
    print(json.dumps(test_events[0], indent=2))

In [ ]:
def send_to_eventstream(events, sb_client, topic_name):
    """Send parsed events to Eventstream via ServiceBus."""
    if not events:
        return 0

    sender = sb_client.get_topic_sender(topic_name)
    with sender:
        batch = sender.create_message_batch()
        for event in events:
            msg = ServiceBusMessage(json.dumps(event))
            try:
                batch.add_message(msg)
            except ValueError:
                # Batch full, send and start new
                sender.send_messages(batch)
                batch = sender.create_message_batch()
                batch.add_message(msg)
        sender.send_messages(batch)

    return len(events)

In [ ]:
# Extract topic name from connection string
import re
topic_match = re.search(r'EntityPath=([^;]+)', connection_string)
if topic_match:
    topic_name = topic_match.group(1)
    print(f"Topic name: {topic_name}")
else:
    raise ValueError("Could not extract EntityPath from connection string")

# Strip EntityPath from connection string for ServiceBusClient
sb_connection = re.sub(r';?EntityPath=[^;]+', '', connection_string)
sb_client = ServiceBusClient.from_connection_string(sb_connection)

In [ ]:
# Main polling loop - runs continuously
print("Starting Trip Updates ingestion loop...")
print(f"Polling every {poll_interval}s from /v2/gtfs/realtime/sydneytrains")
print("Press Stop to halt.\n")

total_events = 0
poll_count = 0

while True:
    try:
        feed = fetch_trip_updates(myapikey)
        events = parse_trip_updates(feed)
        sent = send_to_eventstream(events, sb_client, topic_name)
        total_events += sent
        poll_count += 1

        now = datetime.now(sydney_tz).strftime("%H:%M:%S")
        print(f"[{now}] Poll #{poll_count}: {sent} events sent (total: {total_events:,})")

    except Exception as e:
        print(f"Error: {e}")

    time.sleep(poll_interval)